# Analysis: nanoGPT-LotR

Reads the experiment ledger (`results/runs.csv`) and per-run trajectories
(`results/history/*.csv`) and regenerates every figure used in the README and blog post.
Run the experiments first:

```bash
python experiments/run_scaling_law.py
python experiments/run_ablation.py
```

In [ ]:
import os, glob
import pandas as pd
import matplotlib.pyplot as plt

os.makedirs('results/figures', exist_ok=True)
runs = pd.read_csv('results/runs.csv') if os.path.exists('results/runs.csv') else pd.DataFrame()
print(f'{len(runs)} runs logged')
runs.head()

## 1. Loss curves

Train vs val over training. The val curve flattening while train keeps falling is the
overfitting signature.

In [ ]:
fig, ax = plt.subplots(figsize=(7,4.5))
for path in sorted(glob.glob('results/history/*.csv')):
    name = os.path.basename(path)[:-4]
    h = pd.read_csv(path)
    ax.plot(h['iter'], h['train_loss'], '--', alpha=.6, label=f'{name} train')
    ax.plot(h['iter'], h['val_loss'], '-', label=f'{name} val')
ax.set_xlabel('iteration'); ax.set_ylabel('cross-entropy loss')
ax.set_title('Training vs validation loss'); ax.legend(fontsize=7)
fig.tight_layout(); fig.savefig('results/figures/loss_curves.png', dpi=150)
plt.show()

## 2. Scaling law

Val loss vs parameter count for the `scale_*` runs, log-x. With data fixed at 424K chars,
the curve should bend and flatten as the model becomes data-limited rather than capacity-limited.

In [ ]:
sc = runs[runs['name'].str.startswith('scale_')].sort_values('n_params') if len(runs) else runs
if len(sc):
    fig, ax = plt.subplots(figsize=(6.5,4.5))
    ax.plot(sc['n_params'], sc['final_val_loss'], 'o-', label='val')
    ax.plot(sc['n_params'], sc['final_train_loss'], 's--', alpha=.6, label='train')
    ax.set_xscale('log')
    ax.set_xlabel('parameters'); ax.set_ylabel('final loss')
    ax.set_title('Capacity vs loss (data fixed at 424K chars)'); ax.legend()
    fig.tight_layout(); fig.savefig('results/figures/scaling_law.png', dpi=150)
    plt.show()
else:
    print('run experiments/run_scaling_law.py first')

## 3. Regularisation ablation

Final val loss and the train/val gap as a function of dropout, split by weight decay.

In [ ]:
ab = runs[runs['name'].str.startswith('abl_')].copy() if len(runs) else runs
if len(ab):
    ab['gap'] = ab['final_train_loss'] - ab['final_val_loss']
    fig, axes = plt.subplots(1, 2, figsize=(11,4.2))
    for wd, g in ab.groupby('weight_decay'):
        g = g.sort_values('dropout')
        axes[0].plot(g['dropout'], g['final_val_loss'], 'o-', label=f'wd={wd}')
        axes[1].plot(g['dropout'], g['gap'].abs(), 'o-', label=f'wd={wd}')
    axes[0].set_xlabel('dropout'); axes[0].set_ylabel('final val loss'); axes[0].set_title('Val loss'); axes[0].legend()
    axes[1].set_xlabel('dropout'); axes[1].set_ylabel('|train - val| loss'); axes[1].set_title('Generalisation gap'); axes[1].legend()
    fig.tight_layout(); fig.savefig('results/figures/ablation.png', dpi=150)
    plt.show()
else:
    print('run experiments/run_ablation.py first')